In [ ]:
# 1 Decidir dataset
import pandas as pd
# Corrected URL to raw JSON content
url = 'https://raw.githubusercontent.com/damarals/biblias/main/inst/json/NVT.json'

dfb = pd.read_json(url)

dfb.head()
dfb = dfb[dfb['name'] == 'João'] # Changed 'livro' to 'name'
# Mostrar o DataFrame atualizado
dfb.head()

,abbrev,chapters,name
42,Jo,"[[No princípio, aquele que é a Palavra já exis...",João


In [ ]:
# Accessing the chapters'
joao_chapters = dfb['chapters'].iloc[0]

# Display the first few chapters to see their structure
print(f"Total chapters in João: {len(joao_chapters)}")
print("\nFirst chapter:\n")
print(joao_chapters[0])

print("\nSecond chapter:\n")
print(joao_chapters[1])

Total chapters in João: 21

First chapter:

['No princípio, aquele que é a Palavra já existia. A Palavra estava com Deus, e a Palavra era Deus.', 'Ele existia no princípio com Deus.', 'Por meio dele Deus criou todas as coisas, e sem ele nada foi criado.', 'Aquele que é a Palavra possuía a vida, e sua vida trouxe luz a todos.', 'A luz brilha na escuridão, e a escuridão nunca conseguiu apagá-la.', 'Deus enviou um homem chamado João', 'para falar a respeito da luz, a fim de que, por meio de seu testemunho, todos cressem.', 'Ele não era a luz, mas veio para falar da luz.', 'Aquele que é a verdadeira luz, que ilumina a todos, estava chegando ao mundo.', 'Veio ao mundo que ele criou, mas o mundo não o reconheceu.', 'Veio a seu próprio povo, e eles o rejeitaram.', 'Mas, a todos que creram nele e o aceitaram, ele deu o direito de se tornarem filhos de Deus.', 'Estes não nasceram segundo a ordem natural, nem como resultado da paixão ou da vontade humana, mas nasceram de Deus.', 'Assim, a Palavr

In [ ]:
# 1. Reload the full DataFrame to ensure all books are available
import pandas as pd

url = 'https://raw.githubusercontent.com/damarals/biblias/main/inst/json/NVT.json'
dfb_full = pd.read_json(url)

# 2. Define lists of books for the New Testament and Wisdom books (Old Testament)
new_testament_books = [
    'Mateus', 'Marcos', 'Lucas', 'João', 'Atos', 'Romanos', '1 Coríntios', '2 Coríntios',
    'Gálatas', 'Efésios', 'Filipenses', 'Colossenses', '1 Tessalonicenses', '2 Tessalonicenses',
    '1 Timóteo', '2 Timóteo', 'Tito', 'Filemom', 'Hebreus', 'Tiago', '1 Pedro', '2 Pedro',
    '1 João', '2 João', '3 João', 'Judas', 'Apocalipse'
]

wisdom_books_ot = [
    'Jó', 'Salmos', 'Provérbios', 'Eclesiastes', 'Cânticos'
]

# Combine the lists of desired books
selected_books = new_testament_books + wisdom_books_ot

# 3. Filter the DataFrame for these selected books
dfb_selected = dfb_full[dfb_full['name'].isin(selected_books)].copy()

# 4. Initialize total word count for all selected books
total_words_all_selected_books = 0

print(f"Starting word count for {len(selected_books)} selected books:\n")

# 5. Iterate through each selected book to count words
for index, row in dfb_selected.iterrows():
    book_name = row['name']
    book_chapters_data = row['chapters'] # This could be a complex nested structure
    current_book_word_count = 0

    # Flatten book_chapters_data into a single list of all verse strings
    all_verses_for_book = []

    def flatten_list_of_lists(nested_list):
        for item in nested_list:
            if isinstance(item, list):
                yield from flatten_list_of_lists(item)
            else:
                yield item

    # Apply flattening. This handles arbitrary levels of nesting until strings are found.
    for verse_candidate in flatten_list_of_lists(book_chapters_data):
        if isinstance(verse_candidate, str):
            all_verses_for_book.append(verse_candidate)
        else:
            # This case indicates an unexpected non-string item after deep flattening
            print(f"Warning: Found non-string item in {book_name} after flattening: {verse_candidate} (type: {type(verse_candidate)}) - skipping")

    # Now, all_verses_for_book should be a flat list of strings
    for verse_text in all_verses_for_book:
        words = verse_text.split() # verse_text is guaranteed to be a string here
        current_book_word_count += len(words)

    print(f"  Book: {book_name} - Total words: {current_book_word_count}")
    total_words_all_selected_books += current_book_word_count

print(f"\n------------------------------------------------")
print(f"Total words across all selected books: {total_words_all_selected_books}")


Starting word count for 32 selected books:

  Book: Jó - Total words: 15419
  Book: Salmos - Total words: 36889
  Book: Provérbios - Total words: 13210
  Book: Eclesiastes - Total words: 4724
  Book: Cânticos - Total words: 2449
  Book: Mateus - Total words: 20990
  Book: Marcos - Total words: 13305
  Book: Lucas - Total words: 22346
  Book: João - Total words: 16840
  Book: Atos - Total words: 21569
  Book: Romanos - Total words: 9899
  Book: 1 Coríntios - Total words: 9364
  Book: 2 Coríntios - Total words: 6100
  Book: Gálatas - Total words: 3215
  Book: Efésios - Total words: 2992
  Book: Filipenses - Total words: 2197
  Book: Colossenses - Total words: 2045
  Book: 1 Tessalonicenses - Total words: 1801
  Book: 2 Tessalonicenses - Total words: 1018
  Book: 1 Timóteo - Total words: 2419
  Book: 2 Timóteo - Total words: 1748
  Book: Tito - Total words: 1042
  Book: Filemom - Total words: 430
  Book: Hebreus - Total words: 6792
  Book: Tiago - Total words: 2172
  Book: 1 Pedro - Total

# 1 - Pipeline de Pré-processamento Textual

Etapa para preparar o texto da Bíblia em português.

In [ ]:
# 1. Instalação de dependências e download de modelos
!pip install -q spacy nltk pandas gensim scikit-learn
!python -m spacy download pt_core_news_sm

import spacy
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
import pandas as pd
import re
from collections import Counter
from typing import List, Dict, Any
import json
from pathlib import Path

# Downloads necessários para o NLTK
nltk.download('stopwords')
nltk.download('rslp')

print("\n[INFO] Ambiente configurado com sucesso.")
# O modelo 'pt_core_news_sm' é essencial para o spaCy processar a língua portuguesa,
# permitindo análise morfológica e lemmatização precisa, diferente de modelos em inglês
# que utilizam regras gramaticais e vocabulários distintos.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 33.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

[INFO] Ambiente configurado com sucesso.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


In [ ]:
class BiblePreprocessor:
    """Classe para pré-processamento de texto bíblico focado em Topic Modeling."""

    def __init__(self, custom_stopwords: List[str] = None):
        # Carregando spaCy pt_core_news_sm para análise contextual
        self.nlp = spacy.load("pt_core_news_md")

        # Configuração de Stopwords (NLTK + Custom)
        base_stopwords = set(stopwords.words('portuguese'))
        if custom_stopwords:
            base_stopwords.update(custom_stopwords)
        self.stop_words = base_stopwords

        self.stemmer = RSLPStemmer() # Para comparação acadêmica

    def clean_text(self, text: str) -> str:
        """Limpeza inicial: remoção de ruídos básicos."""
        text = text.lower()
        text = re.sub(r'\d+', '', text) # Remove números
        text = re.sub(r'\s+', ' ', text).strip() # Remove espaços extras
        return text

    def process_pipeline(self, text: str) -> List[str]:
        """
        Executa Tokenização, Lemmatização e POS Filtering.
        Mantém apenas NOUN, VERB, ADJ para densidade semântica no Topic Modeling.
        """
        doc = self.nlp(self.clean_text(text))

        tokens_limpos = []
        for token in doc:
            # POS Filtering: Substantivos, Verbos e Adjetivos carregam o tópico
            if token.pos_ in ['NOUN', 'VERB', 'ADJ']:
                # Lemmatização para reduzir dimensionalidade sem perder semântica
                lemma = token.lemma_
                if lemma not in self.stop_words and len(lemma) > 2:
                    tokens_limpos.append(lemma)

        return tokens_limpos

In [ ]:
!python -m spacy download pt_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 15.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# 2. Execução e Análise Quantitativa
custom_stops = ['capítulo', 'versículo', 'bíblia', 'livro']
preprocessor = BiblePreprocessor(custom_stopwords=custom_stops)

def get_book_text(chapters_data: Any) -> str:
    """Achata a estrutura de capítulos/versículos em uma única string."""
    def flatten(nested):
        for item in nested:
            if isinstance(item, list):
                yield from flatten(item)
            else:
                yield item
    return " ".join([str(v) for v in flatten(chapters_data)])

print(f"Processando {len(dfb_selected)} livros selecionados...")

corpus_final = []
for _, row in dfb_selected.iterrows():
    # Agrega todo o texto do livro de forma robusta
    full_text = get_book_text(row['chapters'])
    tokens = preprocessor.process_pipeline(full_text)
    corpus_final.append(tokens)

print(f"\n[INFO] Corpus processado. Total de documentos: {len(corpus_final)}")
print(f"Exemplo de tokens (primeiros 10 de {dfb_selected['name'].iloc[0]}):", corpus_final[0][:10])

Processando 32 livros selecionados...

[INFO] Corpus processado. Total de documentos: 32
Exemplo de tokens (primeiros 10 de Jó): ['homem', 'chamar', 'viver', 'terra', 'íntegro', 'correto', 'temer', 'deus', 'manter', 'afastar']


In [ ]:
# 3. Análise Quantitativa e Métricas de Impacto

def calculate_metrics(tokens_list: List[str]) -> Dict[str, Any]:
    """Calcula métricas básicas de NLP para análise de densidade lexical."""
    total_tokens = len(tokens_list)
    vocab_size = len(set(tokens_list))
    ttr = vocab_size / total_tokens if total_tokens > 0 else 0
    return {
        "total_tokens": total_tokens,
        "vocab_size": vocab_size,
        "ttr": round(ttr, 4)
    }

# Analisando o impacto (Exemplo com o primeiro livro)
first_book_name = dfb_selected['name'].iloc[0]
raw_text_example = get_book_text(dfb_selected['chapters'].iloc[0])
raw_tokens = raw_text_example.split()

metrics_raw = calculate_metrics(raw_tokens)
metrics_processed = calculate_metrics(corpus_final[0])

# Criando tabela comparativa de impacto
df_impacto = pd.DataFrame({
    "Métrica": ["Total de Tokens", "Tamanho do Vocabulário", "Type-Token Ratio (TTR)"],
    "Original": [metrics_raw['total_tokens'], metrics_raw['vocab_size'], metrics_raw['ttr']],
    "Pós-Processamento": [metrics_processed['total_tokens'], metrics_processed['vocab_size'], metrics_processed['ttr']]
})

print(f"\n[ANÁLISE] Impacto do Pré-processamento no livro: {first_book_name}")
display(df_impacto)

# Justificativa Técnica:
# A redução no tamanho do vocabulário e o ajuste do TTR são fundamentais para LDA/NMF.
# Menos 'ruído' (tokens irrelevantes) permite que o modelo foque em padrões probabilísticos
# de palavras que realmente definem o conteúdo semântico dos tópicos."

# 4. Geração do Corpus Final e Persistência

# Preparando listas para Topic Modeling (Gensim e Sklearn)
# corpus_final já é uma lista de listas de tokens

# Exportação para JSON (Vocabulário)
all_tokens_flat = [t for doc in corpus_final for t in doc]
vocabulario = sorted(list(set(all_tokens_flat)))

output_path = Path("preprocessed_bible")
output_path.mkdir(exist_ok=True)

with open(output_path / "vocabulario.json", "w", encoding="utf-8") as f:
    json.dump(vocabulario, f, ensure_ascii=False, indent=4)

# Salvando corpus preprocessado
with open(output_path / "corpus_final.txt", "w", encoding="utf-8") as f:
    for doc_tokens in corpus_final:
        f.write(" ".join(doc_tokens) + "\n")

print(f"\nPipeline finalizado.")
print(f"- Arquivos salvos em: {output_path}/")
print(f"- Vocabulário final: {len(vocabulario)} termos únicos.")



[ANÁLISE] Impacto do Pré-processamento no livro: Jó


,Métrica,Original,Pós-Processamento
0,Total de Tokens,15419.0000,6668.0000
1,Tamanho do Vocabulário,4836.0000,2151.0000
2,Type-Token Ratio (TTR),0.3136,0.3226



Pipeline finalizado.
- Arquivos salvos em: preprocessed_bible/
- Vocabulário final: 8586 termos únicos.


O português possui muita:

flexão verbal,
composição com pronomes,
hifenização,
variações antigas da linguagem.

Então o vocabulário explode artificialmente.

Exemplo:
    "ajoelhar se",
    "ajoelhar-se",
    "ajoelhavam-se",
    "ajoelhou se",
    "ajoelhou-se",
    "ajoelhour se",
    "ajuda nós",
    "ajuda-a",
    "ajuda-me",
    "ajuda-nos",
    "ajuda-o",
    "ajuda-os",

In [ ]:
mudar de:  para: pt_core_news_md